In [ ]:
# 📦 IMPORTS REQUIS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, auc, roc_auc_score)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configuration pour les visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("✅ Toutes les librairies sont importées avec succès!")

# 🌾 Prédiction des Bonnes et Mauvaises Récoltes au Burundi

## Objectif
Construire un système complet de Machine Learning capable de prédire si une récolte sera **bonne (1)** ou **mauvaise (0)** à partir de données agricoles du Burundi.

## Structure du Projet
- **EDA (Exploratory Data Analysis)** : Analyse complète des données
- **Prétraitement** : Encodage et normalisation
- **Modélisation** : 3 modèles ML (Decision Tree, Random Forest, Logistic Regression)
- **Évaluation** : Métriques complètes et comparaison
- **Optimisation** : Tests d'overfitting et validation croisée
- **Sauvegarde** : Export des modèles en .pkl

## 1️⃣ ANALYSE EXPLORATOIRE DES DONNÉES (EDA)

In [ ]:
# Chargement du dataset
df = pd.read_csv('agriculture_burundi.csv')

# Informations générales sur le dataset
print("=" * 80)
print("📊 DIMENSIONS DU DATASET")
print("=" * 80)
print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}\n")

# Affichage des types de données
print("=" * 80)
print("📋 TYPES DE DONNÉES")
print("=" * 80)
print(df.info())

# Affichage des premières lignes
print("\n" + "=" * 80)
print("👀 APERÇU DES DONNÉES (5 premières lignes)")
print("=" * 80)
print(df.head())

In [ ]:
# Analyse des valeurs manquantes
print("\n" + "=" * 80)
print("❌ ANALYSE DES VALEURS MANQUANTES")
print("=" * 80)

missing_data = pd.DataFrame({
    'Colonne': df.columns,
    'Valeurs_Manquantes': df.isnull().sum(),
    'Pourcentage_%': (df.isnull().sum() / len(df)) * 100
})
print(missing_data.to_string(index=False))

if df.isnull().sum().sum() == 0:
    print("\n✅ Pas de valeurs manquantes détectées!")
else:
    print(f"\n⚠️  Total de valeurs manquantes : {df.isnull().sum().sum()}")

In [ ]:
# Statistiques descriptives par type de variable
print("\n" + "=" * 80)
print("📈 STATISTIQUES DESCRIPTIVES")
print("=" * 80)
print("\n🔢 Variables numériques:")
print(df.describe())

print("\n📝 Variables catégoriques:")
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"  Nombre de catégories: {df[col].nunique()}")

## 2️⃣ VISUALISATIONS OBLIGATOIRES

In [ ]:
# Créer une figure avec plusieurs subplots
fig = plt.figure(figsize=(16, 12))

# 1. Distribution des récoltes (0/1)
ax1 = plt.subplot(2, 3, 1)
bonne_recolte_counts = df['bonne_recolte'].value_counts()
colors = ['#FF6B6B', '#4ECDC4']
ax1.bar(['Mauvaise (0)', 'Bonne (1)'], bonne_recolte_counts.values, color=colors)
ax1.set_ylabel('Nombre de cas', fontsize=11)
ax1.set_title('Distribution des Récoltes (0 = Mauvaise, 1 = Bonne)', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(bonne_recolte_counts.values):
    ax1.text(i, v + 1, str(v), ha='center', fontweight='bold')

# 2. Boxplot rendement par culture
ax2 = plt.subplot(2, 3, 2)
if 'culture' in df.columns and 'rendement' in df.columns:
    df.boxplot(column='rendement', by='culture', ax=ax2)
    ax2.set_title('Rendement par Culture', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Culture', fontsize=11)
    ax2.set_ylabel('Rendement', fontsize=11)
    plt.sca(ax2)
    plt.xticks(rotation=45)

# 3. Heatmap de corrélation
ax3 = plt.subplot(2, 3, 3)
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, ax=ax3, 
            square=True, annot_kws={'size': 8})
ax3.set_title('Matrice de Corrélation', fontsize=12, fontweight='bold')

# 4. Production totale par année (si disponible)
ax4 = plt.subplot(2, 3, 4)
if 'annee' in df.columns and 'production_totale' in df.columns:
    production_by_year = df.groupby('annee')['production_totale'].sum()
    ax4.plot(production_by_year.index, production_by_year.values, marker='o', linewidth=2, markersize=8, color='#2ECC71')
    ax4.set_title('Production Totale par Année', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Année', fontsize=11)
    ax4.set_ylabel('Production (tonnes)', fontsize=11)
    ax4.grid(alpha=0.3)

# 5. Impact engrais vs récolte
ax5 = plt.subplot(2, 3, 5)
if 'engrais' in df.columns:
    engrais_recolte = df.groupby('engrais')['bonne_recolte'].agg(['sum', 'count'])
    engrais_recolte['pourcentage_bon'] = (engrais_recolte['sum'] / engrais_recolte['count']) * 100
    ax5.bar(engrais_recolte.index.astype(str), engrais_recolte['pourcentage_bon'], color='#9B59B6')
    ax5.set_title('Impact de l\'Engrais sur les Bonnes Récoltes (%)', fontsize=12, fontweight='bold')
    ax5.set_xlabel('Quantité d\'Engrais', fontsize=11)
    ax5.set_ylabel('% de Bonnes Récoltes', fontsize=11)
    ax5.grid(axis='y', alpha=0.3)
    for i, v in enumerate(engrais_recolte['pourcentage_bon'].values):
        ax5.text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Visualisations créées avec succès!")

## 3️⃣ PRÉTRAITEMENT DES DONNÉES

In [ ]:
# Copie du dataframe pour manipulation
df_processed = df.copy()

print("=" * 80)
print("🔧 ÉTAPE 1: SUPPRESSION DU DATA LEAKAGE")
print("=" * 80)

# Identifier et supprimer les colonnes causant du data leakage
leakage_cols = []
if 'rendement_t_ha' in df_processed.columns:
    leakage_cols.append('rendement_t_ha')
if 'production_totale_t' in df_processed.columns:
    leakage_cols.append('production_totale_t')

if leakage_cols:
    print(f"❌ Colonnes supprimées (data leakage): {leakage_cols}")
    df_processed = df_processed.drop(columns=leakage_cols, errors='ignore')
else:
    print("✅ Pas de data leakage détecté")

# Identifier les colonnes catégoriques et numériques
categorical_features = df_processed.select_dtypes(include=['object']).columns.tolist()
numeric_features = df_processed.select_dtypes(include=[np.number]).columns.tolist()

# Retirer la variable cible
if 'bonne_recolte' in categorical_features:
    categorical_features.remove('bonne_recolte')
elif 'bonne_recolte' in numeric_features:
    numeric_features.remove('bonne_recolte')

print(f"\n📝 Variables catégoriques à encoder: {categorical_features}")
print(f"🔢 Variables numériques à normaliser: {numeric_features}")

In [ ]:
# Préparation de X et y
print("\n" + "=" * 80)
print("🔧 ÉTAPE 2: SÉPARATION DES FEATURES ET TARGET")
print("=" * 80)

X = df_processed.drop('bonne_recolte', axis=1)
y = df_processed['bonne_recolte']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Distribution de y: \n{y.value_counts()}")

# OneHotEncoder pour les variables catégoriques
print("\n" + "=" * 80)
print("🔧 ÉTAPE 3: ONE-HOT ENCODING")
print("=" * 80)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_categorical_encoded = encoder.fit_transform(X[categorical_features])
X_categorical_encoded_df = pd.DataFrame(
    X_categorical_encoded, 
    columns=encoder.get_feature_names_out(categorical_features)
)

print(f"✅ OneHotEncoding appliqué")
print(f"Nombre de features après encoding: {X_categorical_encoded_df.shape[1]}")

# Combiner variables numériques et catégoriques encodées
X_processed = pd.concat([X[numeric_features].reset_index(drop=True), 
                         X_categorical_encoded_df.reset_index(drop=True)], axis=1)
y_processed = y.reset_index(drop=True)

print(f"\nX_processed shape: {X_processed.shape}")
print(f"Colonnes: {list(X_processed.columns)[:5]}... (et {len(X_processed.columns)-5} autres)")

In [ ]:
# StandardScaler pour les variables numériques
print("\n" + "=" * 80)
print("🔧 ÉTAPE 4: STANDARDISATION (StandardScaler)")
print("=" * 80)

scaler = StandardScaler()
X_scaled = X_processed.copy()
X_scaled[numeric_features] = scaler.fit_transform(X_processed[numeric_features])

print("✅ StandardScaler appliqué aux variables numériques")
print(f"Moyenne après scaling: {X_scaled[numeric_features].mean().mean():.6f}")
print(f"Écart-type après scaling: {X_scaled[numeric_features].std().mean():.6f}")

print("\n✅ Prétraitement terminé!")

## 4️⃣ DIVISION TRAIN-TEST

In [ ]:
# Train-test split (80-20) avec stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_processed, 
    test_size=0.2, 
    stratify=y_processed,
    random_state=42
)

print("=" * 80)
print("📊 DIVISION TRAIN-TEST (80-20 avec stratification)")
print("=" * 80)
print(f"\n✅ Ensemble d'entraînement:")
print(f"   X_train shape: {X_train.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   Distribution: {y_train.value_counts().to_dict()}")

print(f"\n✅ Ensemble de test:")
print(f"   X_test shape: {X_test.shape}")
print(f"   y_test shape: {y_test.shape}")
print(f"   Distribution: {y_test.value_counts().to_dict()}")

print(f"\n📊 Proportion de classes (train):")
print(f"   Mauvaise récolte: {(y_train==0).sum() / len(y_train) * 100:.2f}%")
print(f"   Bonne récolte: {(y_train==1).sum() / len(y_train) * 100:.2f}%")

## 5️⃣ ENTRAÎNEMENT DES MODÈLES

In [ ]:
# 1. DECISION TREE
print("=" * 80)
print("🌳 MODÈLE 1: DECISION TREE")
print("=" * 80)

dt_model = DecisionTreeClassifier(max_depth=4, criterion='gini', random_state=42)
dt_model.fit(X_train, y_train)
print("✅ Decision Tree entraîné (max_depth=4, criterion='gini')")

# 2. RANDOM FOREST
print("\n" + "=" * 80)
print("🌲 MODÈLE 2: RANDOM FOREST")
print("=" * 80)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("✅ Random Forest entraîné (n_estimators=100)")

# 3. LOGISTIC REGRESSION
print("\n" + "=" * 80)
print("📈 MODÈLE 3: LOGISTIC REGRESSION")
print("=" * 80)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print("✅ Logistic Regression entraînée (max_iter=1000)")

print("\n✅ Tous les modèles sont entraînés!")

## 6️⃣ ÉVALUATION COMPLÈTE DES MODÈLES

In [ ]:
# Fonction pour évaluer un modèle
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    print(f"\n{'=' * 80}")
    print(f"📊 ÉVALUATION: {model_name}")
    print(f"{'=' * 80}")
    
    # Prédictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métriques de train
    print(f"\n🎯 PERFORMANCE ENTRAÎNEMENT:")
    train_accuracy = accuracy_score(y_train, y_train_pred)
    print(f"   Accuracy: {train_accuracy:.4f}")
    
    # Métriques de test
    print(f"\n🎯 PERFORMANCE TEST:")
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    
    print(f"   Accuracy:  {test_accuracy:.4f}")
    print(f"   Precision: {test_precision:.4f}")
    print(f"   Recall:    {test_recall:.4f}")
    print(f"   F1-Score:  {test_f1:.4f}")
    print(f"   AUC-ROC:   {test_auc:.4f}")
    
    # Matrice de confusion
    print(f"\n📊 MATRICE DE CONFUSION:")
    cm = confusion_matrix(y_test, y_test_pred)
    print(f"   Vrais Négatifs:  {cm[0, 0]}")
    print(f"   Faux Positifs:   {cm[0, 1]}")
    print(f"   Faux Négatifs:   {cm[1, 0]}")
    print(f"   Vrais Positifs:  {cm[1, 1]}")
    
    return {
        'model_name': model_name,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'auc': test_auc,
        'y_test_pred': y_test_pred,
        'y_test_pred_proba': y_test_pred_proba,
        'cm': cm
    }

# Évaluation des 3 modèles
results_dt = evaluate_model(dt_model, X_train, X_test, y_train, y_test, "Decision Tree")
results_rf = evaluate_model(rf_model, X_train, X_test, y_train, y_test, "Random Forest")
results_lr = evaluate_model(lr_model, X_train, X_test, y_train, y_test, "Logistic Regression")

In [ ]:
# Tableau de comparaison
print("\n" + "=" * 80)
print("📈 TABLEAU DE COMPARAISON DES MODÈLES")
print("=" * 80)

comparison_df = pd.DataFrame({
    'Modèle': [results_dt['model_name'], results_rf['model_name'], results_lr['model_name']],
    'Accuracy': [results_dt['test_accuracy'], results_rf['test_accuracy'], results_lr['test_accuracy']],
    'Precision': [results_dt['precision'], results_rf['precision'], results_lr['precision']],
    'Recall': [results_dt['recall'], results_rf['recall'], results_lr['recall']],
    'F1-Score': [results_dt['f1'], results_rf['f1'], results_lr['f1']],
    'AUC-ROC': [results_dt['auc'], results_rf['auc'], results_lr['auc']]
})

print(comparison_df.to_string(index=False))

# Identifier le meilleur modèle
best_model_idx = comparison_df['Accuracy'].idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Modèle']
print(f"\n🏆 MEILLEUR MODÈLE: {best_model_name} (Accuracy: {comparison_df.loc[best_model_idx, 'Accuracy']:.4f})")

In [ ]:
# Visualisation des matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (results, ax) in enumerate(zip([results_dt, results_rf, results_lr], axes)):
    cm = results['cm']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=True,
                xticklabels=['Mauvaise', 'Bonne'], yticklabels=['Mauvaise', 'Bonne'])
    ax.set_title(f"Matrice de Confusion\n{results['model_name']}", fontweight='bold')
    ax.set_ylabel('Réelle')
    ax.set_xlabel('Prédite')

plt.tight_layout()
plt.show()

print("✅ Matrices de confusion affichées!")

In [ ]:
# Courbes ROC pour chaque modèle
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (results, ax) in enumerate(zip([results_dt, results_rf, results_lr], axes)):
    fpr, tpr, _ = roc_curve(y_test, results['y_test_pred_proba'])
    roc_auc = results['auc']
    
    ax.plot(fpr, tpr, color='#2ECC71', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], color='#E74C3C', lw=2, linestyle='--', label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Faux Positifs')
    ax.set_ylabel('Vrais Positifs')
    ax.set_title(f"Courbe ROC\n{results['model_name']}", fontweight='bold')
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Courbes ROC affichées!")

## 7️⃣ ANALYSE DE L'OVERFITTING

In [ ]:
print("=" * 80)
print("🔍 ANALYSE DE L'OVERFITTING (max_depth de 1 à 20)")
print("=" * 80)

train_accuracies = []
test_accuracies = []
max_depths = range(1, 21)

for depth in max_depths:
    dt_temp = DecisionTreeClassifier(max_depth=depth, criterion='gini', random_state=42)
    dt_temp.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, dt_temp.predict(X_train))
    test_acc = accuracy_score(y_test, dt_temp.predict(X_test))
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

# Visualisation
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(max_depths, train_accuracies, marker='o', linestyle='-', linewidth=2, 
        label='Accuracy Entraînement', color='#3498DB', markersize=6)
ax.plot(max_depths, test_accuracies, marker='s', linestyle='-', linewidth=2, 
        label='Accuracy Test', color='#E74C3C', markersize=6)

ax.set_xlabel('Max Depth', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Analyse de l\'Overfitting: Decision Tree (max_depth 1-20)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(max_depths)

plt.tight_layout()
plt.show()

# Identifier l'overfitting
print("\n📊 RÉSULTATS:")
for depth, train_acc, test_acc in zip(max_depths, train_accuracies, test_accuracies):
    overfitting = train_acc - test_acc
    status = "🔴 OVERFITTING" if overfitting > 0.1 else "✅ OK"
    print(f"Depth {depth:2d}: Train={train_acc:.4f} | Test={test_acc:.4f} | Écart={overfitting:.4f} {status}")

# Trouver la meilleure profondeur
best_depth_idx = np.argmax(test_accuracies)
best_depth = list(max_depths)[best_depth_idx]
print(f"\n🏆 Meilleure profondeur: {best_depth} (Accuracy test: {test_accuracies[best_depth_idx]:.4f})")

## 8️⃣ VALIDATION CROISÉE (5-Fold)

In [ ]:
print("=" * 80)
print("🔄 VALIDATION CROISÉE (5-Fold)")
print("=" * 80)

# Decision Tree Cross-Validation
print("\n🌳 Decision Tree:")
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"   Scores: {cv_scores_dt}")
print(f"   Moyenne: {cv_scores_dt.mean():.4f}")
print(f"   Écart-type: {cv_scores_dt.std():.4f}")
print(f"   Intervalle: [{cv_scores_dt.mean() - cv_scores_dt.std():.4f}, {cv_scores_dt.mean() + cv_scores_dt.std():.4f}]")

# Random Forest Cross-Validation
print("\n🌲 Random Forest:")
cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"   Scores: {cv_scores_rf}")
print(f"   Moyenne: {cv_scores_rf.mean():.4f}")
print(f"   Écart-type: {cv_scores_rf.std():.4f}")
print(f"   Intervalle: [{cv_scores_rf.mean() - cv_scores_rf.std():.4f}, {cv_scores_rf.mean() + cv_scores_rf.std():.4f}]")

# Logistic Regression Cross-Validation
print("\n📈 Logistic Regression:")
cv_scores_lr = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"   Scores: {cv_scores_lr}")
print(f"   Moyenne: {cv_scores_lr.mean():.4f}")
print(f"   Écart-type: {cv_scores_lr.std():.4f}")
print(f"   Intervalle: [{cv_scores_lr.mean() - cv_scores_lr.std():.4f}, {cv_scores_lr.mean() + cv_scores_lr.std():.4f}]")

# Visualisation
fig, ax = plt.subplots(figsize=(10, 6))
models = ['Decision Tree', 'Random Forest', 'Logistic Regression']
means = [cv_scores_dt.mean(), cv_scores_rf.mean(), cv_scores_lr.mean()]
stds = [cv_scores_dt.std(), cv_scores_rf.std(), cv_scores_lr.std()]

ax.bar(models, means, yerr=stds, capsize=10, color=['#9B59B6', '#27AE60', '#F39C12'], 
       alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy (CV=5)', fontsize=12, fontweight='bold')
ax.set_title('Validation Croisée - Comparaison des Modèles', fontsize=13, fontweight='bold')
ax.set_ylim([0.7, 1.0])
ax.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}±{std:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ Validation croisée complétée!")

## 9️⃣ SAUVEGARDE DES MODÈLES

In [ ]:
import os

print("=" * 80)
print("💾 SAUVEGARDE DES MODÈLES ET OBJETS DE PRÉTRAITEMENT")
print("=" * 80)

# Répertoire de base (racine du projet)
base_dir = '.'

# Sauvegarder les modèles
print("\n🔹 Sauvegarde des modèles ML:")

# Decision Tree
joblib.dump(dt_model, os.path.join(base_dir, 'decision_tree.pkl'))
print("   ✅ decision_tree.pkl")

# Random Forest
joblib.dump(rf_model, os.path.join(base_dir, 'random_forest.pkl'))
print("   ✅ random_forest.pkl")

# Logistic Regression
joblib.dump(lr_model, os.path.join(base_dir, 'logistic_regression.pkl'))
print("   ✅ logistic_regression.pkl")

print("\n🔹 Sauvegarde des objets de prétraitement:")

# Scaler
joblib.dump(scaler, os.path.join(base_dir, 'scaler.pkl'))
print("   ✅ scaler.pkl")

# Encoder
joblib.dump(encoder, os.path.join(base_dir, 'encoder.pkl'))
print("   ✅ encoder.pkl")

print("\n✅ Tous les fichiers ont été sauvegardés avec succès!")
print(f"\n📁 Fichiers créés dans: {os.path.abspath(base_dir)}")

# Vérifier l'existence des fichiers
print("\n📂 Vérification des fichiers:")
files_to_check = ['decision_tree.pkl', 'random_forest.pkl', 'logistic_regression.pkl', 
                   'scaler.pkl', 'encoder.pkl']
for file in files_to_check:
    path = os.path.join(base_dir, file)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024  # Taille en KB
        print(f"   ✅ {file} ({size:.2f} KB)")
    else:
        print(f"   ❌ {file} NOT FOUND")

## 🎓 CONCLUSION

### Résumé du Projet
Ce projet de Machine Learning a permis de:
- ✅ Charger et explorer le dataset agricole du Burundi
- ✅ Nettoyer et prétraiter les données
- ✅ Construire 3 modèles de prédiction
- ✅ Évaluer rigoureusement les performances
- ✅ Analyser l'overfitting et la validation croisée
- ✅ Sauvegarder les modèles pour utilisation en production

### Recommandations
1. **Meilleur Modèle**: Utiliser le modèle avec la plus haute accuracy sur le test set
2. **Prédictions**: Les modèles sont prêts à être utilisés pour prédire les récoltes
3. **Amélioration Future**: 
   - Collecter plus de données
   - Tester d'autres hyperparamètres
   - Utiliser ensemble learning
   - Améliorer la qualité des features